# 01 — Segmentation visual audit

Runs Silero segmentation, quantifies profile sensitivity, and overlays saved intervals on decoded waveforms.

Every displayed denominator and paper-facing visual is also saved under `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside paper1_pipeline_rebuilt.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("Visualization outputs:", VIZ_ROOT)


This notebook does not decide that unusual ALS speech is invalid. It visualizes raw, primary, strict-speech, and guarded-nonspeech views and keeps flags separate from hard exclusions.

In [ ]:
RUN_SEGMENTATION = False
if RUN_SEGMENTATION:
    run_cli("segment")
else:
    print("Using existing segmentation outputs.")


In [ ]:
segments = read_stage("01_segmentation/bamboo_segmentation_intervals")
errors = read_stage("01_segmentation/segmentation_errors")

required = {"file_name", "profile", "view", "start_sec", "end_sec", "duration_sec"}
missing = required - set(segments.columns)
assert not missing, f"Segmentation table is missing columns: {sorted(missing)}"
assert (segments["end_sec"] >= segments["start_sec"]).all(), "Negative interval detected"
assert np.allclose(
    segments["duration_sec"],
    segments["end_sec"] - segments["start_sec"],
    rtol=0,
    atol=1e-6,
), "Saved duration does not equal end-start"

summary = (
    segments.groupby(["profile", "view"], as_index=False)
    .agg(
        recordings=("file_name", "nunique"),
        intervals=("duration_sec", "size"),
        total_duration_sec=("duration_sec", "sum"),
        median_interval_sec=("duration_sec", "median"),
        q05_interval_sec=("duration_sec", lambda x: x.quantile(.05)),
        q95_interval_sec=("duration_sec", lambda x: x.quantile(.95)),
    )
)
save_table(summary, "01_segmentation", "interval_summary_by_profile_and_view")
display(summary)
display(errors.head(50))


In [ ]:
# One row per recording/profile: denominators, interval counts, and speech fractions.
recording_duration = (
    segments.groupby(["file_name", "profile"], as_index=False)["end_sec"]
    .max().rename(columns={"end_sec": "recording_duration_sec"})
)
duration_wide = (
    segments.pivot_table(
        index=["file_name", "profile"],
        columns="view",
        values="duration_sec",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()
)
count_wide = (
    segments.pivot_table(
        index=["file_name", "profile"],
        columns="view",
        values="duration_sec",
        aggfunc="size",
        fill_value=0,
    ).add_prefix("n_intervals__").reset_index()
)
recording = recording_duration.merge(duration_wide, on=["file_name", "profile"]).merge(
    count_wide, on=["file_name", "profile"]
)
for view in ["raw_speech", "primary_speech", "strict_speech", "strict_internal_nonspeech"]:
    if view not in recording:
        recording[view] = 0.0
    recording[f"fraction__{view}"] = recording[view] / recording["recording_duration_sec"]
save_table(recording, "01_segmentation", "recording_profile_support")
display(recording.describe(include="all").T)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, view in zip(axes, ["raw_speech", "primary_speech", "strict_speech"]):
    sns.ecdfplot(
        data=recording,
        x=f"fraction__{view}",
        hue="profile",
        ax=ax,
    )
    ax.set(title=view.replace("_", " ").title(), xlabel="Fraction of recording", ylabel="ECDF")
fig.suptitle("Segmentation support across pre-specified profiles", y=1.04)
save_figure(fig, "01_segmentation", "speech_fraction_ecdf")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
primary_counts = recording.loc[recording["profile"].eq("primary")].copy()
count_column = "n_intervals__primary_speech"
sns.histplot(primary_counts[count_column], bins=30, ax=ax)
ax.set(title="Primary speech fragmentation audit", xlabel="Number of primary speech intervals")
save_figure(fig, "01_segmentation", "primary_speech_fragmentation")
plt.show()


In [ ]:
# Waveform overlay for a deliberately editable example.
from paper1_qc.config import load_config, resolve_executable
from paper1_qc.media import decode_audio_views

cfg = load_config(CONFIG)
inventory = read_stage("00_audit/bamboo_media_inventory")
EXAMPLE_FILE = None  # replace with an exact filename, or leave None for the first resolvable file
candidate = EXAMPLE_FILE or segments["file_name"].dropna().iloc[0]
paths = inventory.loc[inventory["file_name"].eq(candidate), "file_path"].tolist()
assert len(paths) == 1, f"Expected one disk path for {candidate}; found {len(paths)}"
audio = decode_audio_views(
    paths[0],
    ffmpeg=resolve_executable(cfg["software"]["ffmpeg"], "ffmpeg"),
    ffprobe=resolve_executable(cfg["software"]["ffprobe"], "ffprobe"),
)
wave = audio.analysis_16k
time = np.arange(len(wave)) / 16000

example_intervals = segments.loc[
    segments["file_name"].eq(candidate) & segments["profile"].eq("primary")
].copy()
palette = {
    "raw_speech": "#4C78A8",
    "primary_speech": "#59A14F",
    "strict_speech": "#F28E2B",
    "strict_internal_nonspeech": "#E15759",
}
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(time, wave, color="0.25", linewidth=0.45, alpha=0.8)
for view, rows in example_intervals.groupby("view"):
    for first, interval in enumerate(rows.itertuples()):
        ax.axvspan(
            interval.start_sec,
            interval.end_sec,
            color=palette.get(view, "0.7"),
            alpha=0.14,
            label=view if first == 0 else None,
        )
ax.set(title=f"Waveform and segmentation views: {candidate}", xlabel="Time (s)", ylabel="Amplitude")
ax.legend(ncol=2, frameon=False)
save_figure(fig, "01_segmentation", "example_waveform_interval_overlay")
plt.show()
